# 🧠 CerebroSpheroid V4: Google Colab GPU Training & Direct Bioflo Cloud Sync

### **CerebroSpheroid V4 Enhancements:**
1. **Modern Llama 32,000 Vocabulary**: Byte-fallback BPE with atomic single-digit arithmetic (`0`-`9`) and zero OOV tokens (118.5M parameters).
2. **Causal Temporal Sensory Ingestion**: Exponential recency decay + phase velocity momentum vectors replacing order-erasing centroid pooling.
3. **Riemannian SpheroidOptimizer**: Persistent power iteration spectral radius guard (<1.5 ms) and single-moment memory saving 50% RAM over AdamW.
4. **Curriculum Scheduled Sampling**: Smoothly transitions from teacher forcing ($p=1.0$) to internal closed-loop deliberation ($p=0.0$) over 10,000 steps.
5. **Direct Bioflo Cloud Sync (Zero Google Drive Quota Consumed)**: Automatically restores latest checkpoint from `bioflo.in` and continuously mirrors lightweight `bfloat16` weights (226 MB) every 500 steps.

---
### ⚡ Colab Settings:
- **Runtime**: **Change runtime type -> T4 GPU** (or A100 GPU on Colab Pro)

## Step 1: GPU Hardware Check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")


## Step 2: Clone / Synchronize Latest Repository & Dependencies

In [ ]:
import os
if not os.path.exists("/content/reasoning_agent"):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git /content/reasoning_agent
%cd /content/reasoning_agent
!git fetch origin main
!git reset --hard origin/main
!pip install -q pyarrow fastparquet transformers datasets sentencepiece accelerate


## Step 3: Run Architecture & Optimizer Acceptance Tests

In [ ]:
!python3 -u training/test_v4_architecture.py
!python3 -u training/test_spheroid_optimizer.py


## Step 4: Run V4 GPU Training (SpheroidOptimizer, Gradient Checkpointing & Bioflo Cloud Sync)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!python3 training/train_v4_full.py \
    --arch v4 \
    --preset v4_small \
    --tokenizer llama \
    --optimizer spheroid \
    --deliberation_mode scheduled \
    --schedule_steps 20000 \
    --min_sampling_p 0.35 \
    --repetition_penalty 1.25 \
    --thalamic_anchor_weight 0.1 \
    --cloud_sync \
    --curriculum \
    --curriculum_stage1_steps 15000 \
    --curriculum_stage1_mix "openhermes=0.75,gsm8k=0.25" \
    --curriculum_stage2_mix "cosmopedia=0.25,fineweb_edu=0.25,finemath=0.15,stack_edu=0.15,openhermes=0.15,gsm8k=0.05" \
    --cloud_url "https://bioflo.in/api/cerebro/sync.php" \
    --cloud_token "CerebroSpheroid_Sync_Secure_2026" \
    --window 128 \
    --eval_window 128 \
    --eval_batch_size 8 \
    --eval_interval 200 \
    --batch_size 16 \
    --grad_accum 2 \
    --lr 0.002 \
    --max_steps 100000 \
    --save_every 500 \
    --save_prefix cerebro_spheroid_v4_colab


## Step 5: Test Model Inference (Questions & Answers)

In [ ]:
import torch
from src.corpus import build_tokenizer
from src.spheroid_v4 import build_spheroid_v4
import glob

ckpts = sorted(glob.glob('checkpoints/*_best.pt') or glob.glob('checkpoints/*.pt'))
if ckpts:
    ckpt_path = ckpts[-1]
    print(f'Loading checkpoint: {ckpt_path}')
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    tok = build_tokenizer('llama')
    model = build_spheroid_v4('v4_small', vocab_size=len(tok))
    model.load_compatible_checkpoint(ckpt_path, device=device)
    model.to(device=device, dtype=torch.float32)
    model.eval()

    questions = [
        'Hi',
        'Hello! Who are you?',
        'What is the capital of France?',
        'What is 2 + 3?',
        'Why is the sky blue?',
        'If you have 3 apples and eat 1, how many are left?',
    ]

    print('\n🧬 Model Inference with 1-Token Biological Delimiters (stim / act):')
    print('=' * 65)
    for q in questions:
        # Generates using 1-token biological framing (stim -> deliberate -> act)
        ans = model.generate_text(
            q,
            tokenizer=tok,
            thinking_cycles=3,
            max_new_tokens=64,
            temperature=0.6,
            repetition_penalty=1.35,
            device=device,
            strip_bio_prefix=True,
        )
        print(f'User:      {q}')
        print(f'Spheroid:  {ans}')
        print('-' * 65)
else:
    print('No checkpoint found in checkpoints/')
